In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
#del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
if os.environ.get("BIRDDOG_USE_LOCAL_NOCODB"):
    print("using local nocodb")
else:
    print("using aws nocodb")

using local nocodb


In [3]:
from birddog.database import Database
from birddog.wiki import (
    page_label,
    sequential_page_label,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
)

2026-07-08 16:29:42,629 [INFO] Using local nocodb api: http://localhost:8080
2026-07-08 16:29:42,779 [INFO] Translation is enabled. Using GCP translator
2026-07-08 16:29:42,779 [INFO] Using Google Cloud translation API
2026-07-08 16:29:42,780 [INFO] GoogleCloudTranslator using REST API


In [11]:
db = Database()

2026-07-08 16:32:16,295 [INFO] creating NocoDBDatabase(host=http://localhost:8080, base_id=p79fvr9cjqgpv5n) instance


In [ ]:
pages = db.get_all_ids("Pages")

In [ ]:
docs = db.get_all_ids("Documents")

In [ ]:
(len(pages), len(docs))

In [ ]:
db.delete("Documents", docs)

In [ ]:
db.delete("Pages", pages)

In [ ]:
r,_ = db.scan("Documents", view_name="BD:WDT:commons.wikimedia.org")

In [8]:
db._field_id("Documents", "owning_pages")

'c8hehahscwgq5rm'

In [13]:
db._field_id("Pages", "parent")

'cu0g321ru7wpv7s'

In [10]:
rename_field(db, db._field_id("Pages", "Pages"), "parent")

In [ ]:
len(pages)

In [ ]:
db.delete("Pages", pages)

In [ ]:
docs = db.get_all_ids("Documents")

In [ ]:
len(docs)

In [ ]:
db.delete("Documents", docs)

In [ ]:
pages = []
cursor = None
while True:
    if cursor and (int(cursor) % 10000) == 0:
        print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=1000,
        where=("seq_label", "is", None),
        fields=("Id","title","label","seq_label"),
    )
    if batch:
        pages.extend(batch)
    if not cursor:
        break

In [ ]:
len(pages)

In [ ]:
pages[1000]

In [ ]:
def normalize_labels(page):
    result = page.copy()
    title = page.get("title")
    if title:
        proper_label = page_label(title)
        result["label"] = proper_label
        proper_seq_label = sequential_page_label(proper_label)
        result["seq_label"] = proper_seq_label
    return result

In [ ]:
normalize_labels(pages[2000])

In [ ]:
pages[2000]

In [ ]:
pages[2000] == normalize_labels(pages[2000])

In [ ]:
pages[2000] == pages[2000].copy()

In [ ]:
norm_pages = [normalize_labels(p) for p in pages]

In [ ]:
norm_pages[:10]

In [ ]:
changed_pages = [n for n,p in zip(norm_pages, pages) if n != p]

In [ ]:
len(changed_pages)

In [ ]:
len(pages)

In [ ]:
len(norm_pages)

In [ ]:
rec_ids = db.write("Pages", norm_pages[1000:2000])

In [ ]:
chunk = 1000
for i in range(0, len(norm_pages), chunk):
    print(i)
    rec_ids = db.write("Pages", norm_pages[i:(i+chunk)])